# tf_to_dict.ipynb
***example of interpreting saved tensorflow model as a python dictionary***
___


## imports

In [1]:
import tensorflow as tf
import json

from tf_to_dict import tf_to_dict
import pickle
import numpy as np

2025-10-03 11:09:30.563338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759486170.574370  583709 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759486170.577753  583709 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-03 11:09:30.590116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## tutorial
here we'll load in an example tensorflow model, `pitchfork.h5`, from the `models` directory and use `tf_to_dict` to unpack the important information (like layer weights and biases, activation functions, branching architecture etc.) and save to a python dictionary

In [2]:
model_name = 'M4-test'

this model was trained using custom objects, which tensorflow specifically needs to be redefined when we try to load in the saved model - so we'll have to define those here before we can even use the model (don't worry about these, you'll know if you need to define your custom objects properly because tensorflow will complain)

In [3]:
class InversePCA(tf.keras.layers.Layer):
    """
    Inverse PCA layer for tensorflow neural network
    
    Usage:
        - Define dictionary of custom objects containing Inverse PCA
        - Use arguments of PCA mean and components from PCA of output parameters for inverse PCA (found in JSON dict)
        
    Example:

    > f = open("pcann_info.json")
    >
    > data = json.load(f)
    >
    > pca_comps = np.array(data["pca_comps"])
    > pca_mean = np.array(data["pca_mean"])
    > 
    > custom_objects = {"InversePCA": InversePCA(pca_comps, pca_mean)}
    > pcann_model = tf.keras.models.load_model("pcann_name.h5", custom_objects=custom_objects)
    
    """
    
    def __init__(self, pca_comps, pca_mean, **kwargs):
        super(InversePCA, self).__init__()
        self.pca_comps = pca_comps
        self.pca_mean = pca_mean
        
    def call(self, x):
        y = tf.tensordot(x, np.float32(self.pca_comps),1) + np.float32(self.pca_mean)
        return y
    
    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'pca_comps': self.pca_comps,
            'pca_mean': self.pca_mean
        })
        return config

class WMSE(tf.keras.losses.Loss):
    """
    Weighted Mean Squared Error Loss Function for tensorflow neural network
    
    Usage:
        - Define list of weights with len = labels
        - Use weights as arguments - no need to square, this is handled in-function
        - Typical usage - defining target precision on outputs for the network to achieve, weights parameters in loss calculation to force network to focus on parameters with unc >> weight.
    
    """
    
    def __init__(self, weights, name = "WMSE",**kwargs):
        super(WMSE, self).__init__()
        self.weights = np.float32(weights)
        
    def call(self, y_true, y_pred):
        loss = ((y_true - y_pred)/(self.weights))**2
        return tf.math.reduce_mean(loss)
    
    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'weights': self.weights
        })
        return config

def WMSE_metric(y_true, y_pred):
    metric = ((y_true - y_pred)/(weights))**2
    return tf.reduce_mean(metric)

with open(f'models/{model_name}_info.pkl', 'rb') as fp:
    model_info = pickle.load(fp)

custom_objects = {
    "WMSE": WMSE_metric,
}

In [4]:
custom_objects

{'WMSE': <function __main__.WMSE_metric(y_true, y_pred)>}

now we can load in the model using `tf.keras.models.load_model` and pass the custom objects to stop tensorflow from whining (again, you can remove the custom_objects keyword if you don't use any - you'll know if you need them by this point)

In [5]:
tf_model = tf.keras.models.load_model(
    f'models/{model_name}.keras', # <- change to keras.model if that's how you saved it!
    custom_objects = custom_objects,
)

I0000 00:00:1759486177.490573  583709 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1212 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6
I0000 00:00:1759486177.491166  583709 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 18257 MB memory:  -> device: 1, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


now we have our model loaded in, we can simply pass this to `tf_to_dict` and wave goodbye to tensorflow!

In [6]:
wtf_dict = tf_to_dict(tf_model)

---
wtf: converting tensorflow model to dict!
---
wtf: finding model info...
	found optimiser: Adam
	found learning rate: 9.999995199905243e-06
---
wtf: finding model layers...
	found input_layer
	found dense
	found dense_1
	found dense_2
	found dense_3
	found dense_4
	found dense_5
	found dense_6
---
wtf: populating dict with outbound_layers:
	input_layer-->dense
	dense-->dense_1
	dense_1-->dense_2
	dense_2-->dense_3
	dense_3-->dense_4
	dense_4-->dense_5
	dense_5-->dense_6
	dense_6 has no outbound layers - output layer?
---
wtf: adding detected network structure to dict:
	input_layer
	  |
	  v
	dense
	  |
	  v
	dense_1
	  |
	  v
	dense_2
	  |
	  v
	dense_3
	  |
	  v
	dense_4
	  |
	  v
	dense_5
	  |
	  v
	dense_6
---
wtf: done!


this function returns a fully hashable python dictionary of weights, biases, activation functions, layer orders etc.

it also prints out the interpretation of the network structure - you should definitely check this to make sure it matches what you're expecting!

the dictionary is structured in a way that is easily interpreted by our compile functions (see `compile_from_dict.ipynb`), but should retain *some* degree of human readability.

let's take a look:

In [13]:
wtf_dict['layers']['dense_1']['activation']

'elu'

once defined, this wtf_dict can be used directly by the compile_from_dict functions we'll look at in `compile_from_dict`, but it's probably best practice to save it as a json file and then we can load this in in a different environment:

In [46]:
with open(f'models/{model_name}.json', 'w') as fp:
    json.dump(wtf_dict, fp)

the idea here is that in our tensorflow training environment (where we have tensorflow installed to train models), you'd save a model using tensorflow as usual, and then immediately `tf_to_dict` the saved model and save as a json file to be used in a different environment where you no longer need tensorflow to be installed